# Zero-DCE — Zero-Reference Deep Curve Estimation / 영-참조 심층 곡선 추정 구현

**Paper**: Guo, C., Li, C., Guo, J., Loy, C. C., Hou, J., Kwong, S., Cong, R., "Zero-Reference Deep Curve Estimation for Low-Light Image Enhancement", *CVPR* 2020. DOI: 10.1109/CVPR42600.2020.00185

이 노트북은 **소형 Zero-DCE** 를 처음부터 구현한다. 합성 저조도 영상(scikit-image astronaut/cameraman을 감마 어둡힘)을 사용하고, 4개의 비참조 손실 (spatial consistency, exposure control, color constancy, illumination smoothness)로 학습한다. 학습 ≤ 200 iterations, image 96×96, CPU 5분 이내.

1. 합성 저조도 데이터셋 (감마 어둡힘 multi-image)
2. DCE-Net (7-layer, 32 channels, symmetric concat)
3. LE-curve 반복 적용
4. 4가지 비참조 손실 정의
5. 학습 루프
6. 결과 시각화 및 평가

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from skimage import data
from skimage.transform import resize
from typing import Tuple

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = torch.device('cpu')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## Part 1: Synthetic low-light dataset / 합성 저조도 데이터

scikit-image의 `astronaut`, `coffee`, `cat`을 96×96 RGB로 resize하고 random gamma ($\gamma \in [2.5, 4.5]$)로 어둡힌다. **Reference 영상은 사용하지 않음** — Zero-DCE는 zero-reference.

In [ ]:
def darken_rgb(img: np.ndarray, gamma: float) -> np.ndarray:
    """Apply gamma darkening to a [0,1] RGB image.

    Args:
        img: Image array (H, W, 3) in [0, 1].
        gamma: Gamma exponent (>1 darkens).

    Returns:
        Darkened image clipped to [0, 1].
    """
    return np.clip(np.power(np.clip(img, 0, 1), gamma), 0, 1).astype(np.float32)


def load_rgb(name: str, size: int = 96) -> np.ndarray:
    """Load an scikit-image sample, ensure RGB, and resize."""
    fn = getattr(data, name)
    img = fn()
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = img.astype(np.float32)
    if img.max() > 1.5:
        img /= 255.0
    img = resize(img, (size, size, 3), anti_aliasing=True).astype(np.float32)
    return np.clip(img, 0, 1)


size = 96
bases = [load_rgb('astronaut', size), load_rgb('coffee', size), load_rgb('cat', size)]
darks = []
for b in bases:
    for gamma in [2.5, 3.0, 3.5, 4.0, 4.5]:
        darks.append(darken_rgb(b, gamma))
darks = np.stack(darks, axis=0)
print('dark stack shape:', darks.shape)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, im in zip(axes[0], bases + [bases[0]]):
    ax.imshow(im)
    ax.axis('off')
    ax.set_title('Reference (NOT used in training)')
for ax, im in zip(axes[1], darks[::4]):
    ax.imshow(im)
    ax.axis('off')
    ax.set_title(f'Synthetic low-light')
plt.tight_layout()
plt.show()

## Part 2: DCE-Net architecture / 네트워크 구조

7-layer plain CNN with 32 channels, ReLU, symmetric concatenation between layers (1↔7, 2↔6, 3↔5). Output: 24 maps = 8 iterations × 3 RGB. Tanh activation.

In [ ]:
class DCENet(nn.Module):
    """DCE-Net: 7-layer CNN that estimates curve-parameter maps.

    Args:
        n_iters: Number of LE-curve iterations (output 3 * n_iters channels).
        n_features: Number of feature channels per conv layer.
    """

    def __init__(self, n_iters: int = 8, n_features: int = 32):
        super().__init__()
        self.n_iters = n_iters
        c = n_features
        self.conv1 = nn.Conv2d(3, c, 3, padding=1)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1)
        self.conv3 = nn.Conv2d(c, c, 3, padding=1)
        self.conv4 = nn.Conv2d(c, c, 3, padding=1)
        # Symmetric concat skip: in-channels become 2*c
        self.conv5 = nn.Conv2d(2 * c, c, 3, padding=1)
        self.conv6 = nn.Conv2d(2 * c, c, 3, padding=1)
        self.conv7 = nn.Conv2d(2 * c, 3 * n_iters, 3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x1))
        x3 = F.relu(self.conv3(x2))
        x4 = F.relu(self.conv4(x3))
        x5 = F.relu(self.conv5(torch.cat([x3, x4], dim=1)))
        x6 = F.relu(self.conv6(torch.cat([x2, x5], dim=1)))
        x7 = torch.tanh(self.conv7(torch.cat([x1, x6], dim=1)))
        return x7  # (B, 3*n_iters, H, W) in [-1, 1]


def apply_le_curves(image: torch.Tensor, maps: torch.Tensor, n_iters: int = 8) -> torch.Tensor:
    """Apply the high-order pixel-wise LE-curve `n_iters` times.

    Args:
        image: Input RGB image (B, 3, H, W) in [0, 1].
        maps: Curve parameter maps (B, 3*n_iters, H, W) in [-1, 1].
        n_iters: Number of iterations.

    Returns:
        Enhanced image (B, 3, H, W).
    """
    y = image
    for n in range(n_iters):
        a = maps[:, 3*n:3*(n+1), :, :]
        y = y + a * y * (1 - y)
    return y


model = DCENet(n_iters=8, n_features=32).to(device)
print(model)
print(f'Trainable params: {sum(p.numel() for p in model.parameters())}')

## Part 3: Non-reference losses / 비참조 손실 4종

In [ ]:
def spatial_consistency_loss(y: torch.Tensor, x: torch.Tensor, region: int = 4) -> torch.Tensor:
    """Loss enforcing that neighbour-region intensity differences are preserved.

    Args:
        y: Enhanced image (B, 3, H, W) in [0, 1].
        x: Original image (B, 3, H, W) in [0, 1].
        region: Pooling region size used to define local averages.

    Returns:
        Scalar loss.
    """
    yg = y.mean(dim=1, keepdim=True)
    xg = x.mean(dim=1, keepdim=True)
    yp = F.avg_pool2d(yg, region)
    xp = F.avg_pool2d(xg, region)
    # Differences with right and below neighbours
    dyx = yp[:, :, :, 1:] - yp[:, :, :, :-1]
    dxx = xp[:, :, :, 1:] - xp[:, :, :, :-1]
    dyy = yp[:, :, 1:, :] - yp[:, :, :-1, :]
    dxy = xp[:, :, 1:, :] - xp[:, :, :-1, :]
    return ((dyx.abs() - dxx.abs()) ** 2).mean() + ((dyy.abs() - dxy.abs()) ** 2).mean()


def exposure_loss(y: torch.Tensor, target: float = 0.6, region: int = 16) -> torch.Tensor:
    """Loss driving the mean intensity of local regions toward `target`."""
    yg = y.mean(dim=1, keepdim=True)
    yp = F.avg_pool2d(yg, region)
    return torch.abs(yp - target).mean()


def color_constancy_loss(y: torch.Tensor) -> torch.Tensor:
    """Gray-world color-constancy loss across RGB channels."""
    means = y.mean(dim=[2, 3])  # (B, 3)
    R, G, B = means[:, 0], means[:, 1], means[:, 2]
    return ((R - G) ** 2 + (R - B) ** 2 + (G - B) ** 2).mean()


def smoothness_loss(maps: torch.Tensor) -> torch.Tensor:
    """Total variation on the curve parameter maps."""
    dx = maps[:, :, :, 1:] - maps[:, :, :, :-1]
    dy = maps[:, :, 1:, :] - maps[:, :, :-1, :]
    return (dx ** 2).mean() + (dy ** 2).mean()

## Part 4: Training loop / 학습

$L_{total} = L_{spa} + L_{exp} + 0.5 L_{col} + 20 L_{tv}$.

Adam $lr=2 \times 10^{-4}$, ≤ 200 iterations on the small synthetic stack. **Reference 영상 사용 없음**.

In [ ]:
X = torch.from_numpy(darks.transpose(0, 3, 1, 2)).float()  # (N, 3, H, W)
ds = TensorDataset(X)
dl = DataLoader(ds, batch_size=4, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=2e-4)
MAX_ITER = 200
iters = 0
history = []
model.train()
while iters < MAX_ITER:
    for (xb,) in dl:
        if iters >= MAX_ITER:
            break
        xb = xb.to(device)
        maps = model(xb)
        y = apply_le_curves(xb, maps, n_iters=8)
        L_spa = spatial_consistency_loss(y, xb)
        L_exp = exposure_loss(y, target=0.6)
        L_col = color_constancy_loss(y)
        L_tv = smoothness_loss(maps)
        loss = L_spa + L_exp + 0.5 * L_col + 20.0 * L_tv
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append((float(L_spa), float(L_exp), float(L_col), float(L_tv), float(loss)))
        iters += 1

history = np.array(history)
fig, ax = plt.subplots(figsize=(8, 3.5))
for col, lbl in zip(range(5), ['L_spa', 'L_exp', 'L_col', 'L_tv', 'L_total']):
    ax.plot(history[:, col], label=lbl, alpha=0.8)
ax.set_yscale('log')
ax.set_xlabel('iteration')
ax.set_title('Zero-DCE training losses')
ax.legend(ncol=5, fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 5: Inference and visualisation / 추론 및 시각화

학습된 DCE-Net을 dark 영상에 적용하고, reference (학습에 사용 안 함)와 비교.

In [ ]:
model.eval()
with torch.no_grad():
    maps = model(X.to(device))
    Y = apply_le_curves(X.to(device), maps, n_iters=8).cpu().numpy()
Y = np.clip(Y.transpose(0, 2, 3, 1), 0, 1)

indices = [0, 5, 10]
fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for row, idx in enumerate(indices):
    base_idx = idx // 5
    axes[row, 0].imshow(bases[base_idx])
    axes[row, 0].set_title('Reference (unused)')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(darks[idx])
    axes[row, 1].set_title(f'Dark input (gamma={[2.5,3.0,3.5,4.0,4.5][idx % 5]})')
    axes[row, 1].axis('off')
    axes[row, 2].imshow(Y[idx])
    axes[row, 2].set_title('Zero-DCE output')
    axes[row, 2].axis('off')
plt.tight_layout()
plt.show()

## Part 6: Curve-parameter map visualisation / 곡선 매개변수 맵 시각화

8 iteration의 RGB 매개변수 맵을 평균하여 표시 (논문 Figure 3과 유사).

In [ ]:
with torch.no_grad():
    sample = X[10:11].to(device)
    maps_sample = model(sample).cpu().numpy()[0]  # (24, H, W)
# Average over iterations -> 3 (RGB) maps
maps_rgb = maps_sample.reshape(8, 3, sample.shape[2], sample.shape[3]).mean(axis=0)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(darks[10])
axes[0].set_title('Input')
axes[0].axis('off')
for i, c in enumerate(['R', 'G', 'B']):
    im = axes[i + 1].imshow(maps_rgb[i], cmap='RdBu_r', vmin=-1, vmax=1)
    axes[i + 1].set_title(f'$\\bar{{\\mathcal{{A}}}}^{c}$')
    axes[i + 1].axis('off')
    plt.colorbar(im, ax=axes[i+1], fraction=0.04)
axes[4].imshow(Y[10])
axes[4].set_title('Output')
axes[4].axis('off')
plt.tight_layout()
plt.show()

## Part 7: Quantitative check vs reference / 정량 확인

Zero-DCE는 학습에 reference를 쓰지 않지만, 평가 단계에서는 참고용으로 PSNR/SSIM을 reference 영상과 비교할 수 있다.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

rows = []
for k in range(len(darks)):
    base = bases[k // 5]
    ref = base
    out = Y[k]
    inp = darks[k]
    psnr_in = psnr_fn(ref, inp, data_range=1.0)
    psnr_out = psnr_fn(ref, out, data_range=1.0)
    ssim_in = ssim_fn(ref, inp, channel_axis=-1, data_range=1.0)
    ssim_out = ssim_fn(ref, out, channel_axis=-1, data_range=1.0)
    rows.append((k, [2.5,3.0,3.5,4.0,4.5][k % 5],
                 psnr_in, psnr_out, ssim_in, ssim_out))
print(f'{"idx":>3} {"gamma":>5} {"PSNR_in":>8} {"PSNR_out":>8} {"SSIM_in":>8} {"SSIM_out":>8}')
for r in rows:
    print(f'{r[0]:>3} {r[1]:>5.1f} {r[2]:>8.2f} {r[3]:>8.2f} {r[4]:>8.3f} {r[5]:>8.3f}')

## Summary / 요약

| Concept / 개념 | This paper / 이 논문 | Modern equivalent / 현대 등가 |
|---|---|---|
| Reference data / 참조 데이터 | none — purely loss-driven | Zero-DCE++, SCI, RUAS (zero-reference family) |
| Output formulation / 출력 형식 | curve parameter map (24 channels) | direct enhancement map (RetinexNet, MBLLEN) |
| Loss / 손실 | $L_{spa}$ + $L_{exp}$ + $L_{col}$ + $L_{tv\mathcal{A}}$ | reconstruction + perceptual + adversarial |
| Iterations / 반복 | 8 (higher-order curve) | unrolled deep network |
| Network size / 네트워크 크기 | 79K params, 5.21G FLOPs | 10K (Zero-DCE++); 100K-10M (others) |
| Inference speed / 속도 | ~500 FPS on GPU 256×256 | typically 10-100 FPS |

**Key takeaway**: 좋은 손실 4가지만 잘 정의하면 reference 없이도 deep network가 의미 있는 enhancement를 학습한다. 작은 합성 데이터셋과 200 iteration만으로 Zero-DCE의 핵심 동작을 재현할 수 있다.

Four well-designed non-reference losses are sufficient to train a deep enhancer — no reference image required. Even a small synthetic stack and 200 iterations reproduce the core behaviour of Zero-DCE.